# 04 — Morphology Features (D-03, canonical)

> **Phase 3 · SSOT §12.3 D-03 Morphology Measurement**
> **Canonical artifact**: `data/registry/MORPH_FEATURES_KIWI_v001.parquet` (SSOT §38 naming)

> **Lineage authority**: Claude-B independent forensic adjudication
> `441d5802bfebe178fd220d08b653c60dfad17faf`
> `ssot/2026-08-17_1730_KOEN_TP_G2_G3_G4_FINAL_ADJUDICATION.md`
> `G2_REPRESENTATION_INTEGRITY_PASS` · `G3_TOKENIZER_INTEGRITY_PASS` · `G4_MORPHOLOGY_INTEGRITY_PASS`
> `MEASUREMENT_FOUNDATION_CLOSED_THROUGH_G4`
>
> Every artifact identity below is pinned to values Claude-B recomputed from the physical Parquet,
> not to values this pipeline reported about itself.


## Scope

Korean morphological structure of the final cohort: morpheme segmentation, the three POS buckets
frozen in SSOT §12.3, the §13.6 density and the §13.7 ratios. Tokenizer/BPE boundaries, the full POS
inventory, and domain-specific dictionaries are outside this notebook (SSOT §15.2).

## What this artifact supersedes

An earlier N=1,000 pilot and its 85-row sanity sample were produced under five conformance defects
and are marked `SUPERSEDED_BY_CONFORMANCE_DEFECT`. They are **not** reachable through the canonical
path and may not be cited as G4 evidence.

| finding | defect | correction |
|---|---|---|
| M1 | `morpheme_density` divided by codepoint count | SSOT §13.6 `MorphemeCount / EojeolCount` |
| M2 | derivational affix matched `XSN`/`XSV`/`XSA` exactly, dropping every irregular variant | `base_tag = tag.partition("-")[0]` |
| M3 | 11 columns; D-03 required fields missing | full 19-column D-03 schema |
| M4 | zero-morpheme rows silently used denominator 1 | pair kept, warning flagged, ratios null |
| M5 | superseded population constant hard-coded | derived from the input artifact |

M2 is the only one that changed a stored number. Its full-cohort effect is isolated in section 5.

## Fail-closed lineage

This notebook **validates and reuses**; it does not rebuild. Canonical inputs are asserted through
`tokenization_premium.lineage.assert_canonical_artifact`, which checks path, SHA-256, column count,
row count and identifier uniqueness, and raises `CanonicalArtifactIdentityMismatch` on any
difference. There is no `if exists()` guard and no descent to a pilot, synthetic, or earlier
version: a missing or altered artifact stops the notebook rather than silently changing what is
being reported.

## 0 — Environment and rebuild gate

In [1]:
from __future__ import annotations

import json

import duckdb
import pandas as pd

from tokenization_premium.lineage import (
    ADJUDICATION_COMMIT,
    ADJUDICATION_DOC,
    CANONICAL_ARTIFACTS,
    CANONICAL_PAIR_SET_MD5,
    assert_canonical_artifact,
    describe_historical,
)
from tokenization_premium.paths import PROJECT_ROOT

pd.set_option("display.width", 200, "display.max_columns", 50)


# 모든 전집단 질의는 DuckDB pushdown으로 처리해 메모리를 bounded 상태로 유지한다.
def connect() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("SET memory_limit='5GB'")
    con.execute("SET threads=8")
    con.execute("SET preserve_insertion_order=false")
    spill = PROJECT_ROOT / ".runtime" / "canonical-nb" / "duckdb-spill"
    spill.mkdir(parents=True, exist_ok=True)
    con.execute(f"SET temp_directory='{spill.as_posix()}'")
    return con


CON = connect()
print(f"lineage authority : {ADJUDICATION_COMMIT[:12]}  {ADJUDICATION_DOC}")

lineage authority : 441d5802bfeb  ssot/2026-08-17_1730_KOEN_TP_G2_G3_G4_FINAL_ADJUDICATION.md


In [2]:
REBUILD_CANONICAL_ARTIFACT = False   # Director-authorized rebuild only; Run All must never regenerate

if REBUILD_CANONICAL_ARTIFACT:
    raise RuntimeError(
        "REBUILD_CANONICAL_ARTIFACT=True는 Director 승인 실행에서만 사용한다. "
        "기본 Run All은 canonical artifact를 재생성하지 않고 검증·재사용만 한다."
    )
print("rebuild gate: DISABLED (validate/reuse only)")

rebuild gate: DISABLED (validate/reuse only)


## 1 — Canonical input and output identity (fail-closed)

The D-02 input supplies `ko_eojeol_count`, the §13.6 denominator. Both artifacts are asserted before
use, including the sorted pair-set hash Claude-B verified independently.

In [3]:
REP = assert_canonical_artifact("REP_FEATURES_v002", verify_pair_set=True, con=CON)
MORPH = assert_canonical_artifact("MORPH_FEATURES_KIWI_v001", verify_pair_set=True, con=CON)

for confirmed in (REP, MORPH):
    print(f"{confirmed['name']:26s} {confirmed['identity']}")
    print(f"   sha256 {confirmed['sha256']}")
    print(f"   rows   {confirmed['row_count']:>9,}   columns {confirmed['column_count']:>3}"
          f"   distinct {confirmed['id_column']} {confirmed['distinct_id']:,}")
    print(f"   pair-set md5 {confirmed['pair_set_md5']}")
assert REP["pair_set_md5"] == MORPH["pair_set_md5"] == CANONICAL_PAIR_SET_MD5

M = f"read_parquet('{CANONICAL_ARTIFACTS['MORPH_FEATURES_KIWI_v001'].path.as_posix()}')"
R = f"read_parquet('{CANONICAL_ARTIFACTS['REP_FEATURES_v002'].path.as_posix()}')"
N = MORPH["row_count"]
print(f"\ncohort N = {N:,} (derived from the artifact, not hard-coded)")

REP_FEATURES_v002          CANONICAL_ARTIFACT_IDENTITY_VERIFIED
   sha256 dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309
   rows   3,835,988   columns  49   distinct pair_id 3,835,988
   pair-set md5 d9660d654ee449e4d0c23a0070225274
MORPH_FEATURES_KIWI_v001   CANONICAL_ARTIFACT_IDENTITY_VERIFIED
   sha256 0fe5bd74e3993a7141c5c33ea78e71b2c66e3ecd296544bde2615acb43e50f7d
   rows   3,835,988   columns  19   distinct morph_measurement_id 3,835,988
   pair-set md5 d9660d654ee449e4d0c23a0070225274

cohort N = 3,835,988 (derived from the artifact, not hard-coded)


## 2 — Superseded pilot (not current evidence)

Recorded for provenance. No cell reads it.

In [4]:
pilot = describe_historical("MORPH_FEATURES_PILOT_v001")
print(json.dumps(pilot, ensure_ascii=False, indent=2))
print("\nThe 85-row sanity sample drawn from this pilot is superseded by the Director N=100 audit.")

{
  "relative_path": ".runtime/nb04-pilot/MORPH_FEATURES_PILOT_v001.parquet",
  "sha256": "41a28f4444e1b346dd3948c106fe1ef34978e6eabcb1216be5a56ed0b833ac31",
  "status": "SUPERSEDED_BY_CONFORMANCE_DEFECT",
  "reason": "M1 codepoint 분모 / M2 exact-match 접사 매핑 하에서 생성됨. G4 증거로 인용 불가",
  "superseded_by": "MORPH_FEATURES_KIWI_v001",
  "present_locally": "False",
  "NOT_CURRENT_EVIDENCE": "true"
}

The 85-row sanity sample drawn from this pilot is superseded by the Director N=100 audit.


## 3 — Analyzer provenance freeze

SSOT §15.1 fixes Kiwi as the primary analyzer and §31 G4 requires the analyzer version and config
hash to be recorded. Every row must carry the same frozen provenance.

In [5]:
from tokenization_premium.morphology import (
    ANALYZER_MODEL_MANIFEST_SHA256,
    MORPHOLOGY_CONFIG,
    MORPHOLOGY_CONFIG_SHA256,
)

PROV_SQL = (
    "SELECT count(DISTINCT analyzer_name) AS analyzer_name,"
    "       count(DISTINCT analyzer_package_version) AS package_version,"
    "       count(DISTINCT analyzer_model_version) AS model_version,"
    "       count(DISTINCT analyzer_config_hash) AS config_hash,"
    "       any_value(analyzer_name) AS name, any_value(analyzer_package_version) AS pkg,"
    "       any_value(analyzer_model_version) AS model, any_value(analyzer_config_hash) AS cfg"
    f" FROM {M}")
prov = CON.execute(PROV_SQL).fetchdf().iloc[0]
print(f"distinct values over {N:,} rows -> name {prov['analyzer_name']}, package "
      f"{prov['package_version']}, model {prov['model_version']}, config {prov['config_hash']}")
print(f"  analyzer           {prov['name']} {prov['pkg']} / model {prov['model']}")
print(f"  artifact config    {prov['cfg']}")
print(f"  module config      {MORPHOLOGY_CONFIG_SHA256}")
print(f"  model manifest sha {ANALYZER_MODEL_MANIFEST_SHA256}")
assert all(int(prov[k]) == 1
           for k in ("analyzer_name", "package_version", "model_version", "config_hash"))
assert prov["cfg"] == MORPHOLOGY_CONFIG_SHA256, "artifact config hash != current module config"
print("\nANALYZER_PROVENANCE_FREEZE = PASS")
print(json.dumps({k: MORPHOLOGY_CONFIG[k] for k in
                  ("morpheme_density_denominator", "particle_tag_rule", "ending_tag_rule",
                   "deriv_affix_rule", "zero_morpheme_policy")}, ensure_ascii=False, indent=1))

distinct values over 3,835,988 rows -> name 1, package 1, model 1, config 1
  analyzer           Kiwi 0.23.2 / model 0.23.0
  artifact config    6f48802a07d984cab18c5cf6c1df2a3e3e778f1825dd0be17438d3d4fb48f23d
  module config      6f48802a07d984cab18c5cf6c1df2a3e3e778f1825dd0be17438d3d4fb48f23d
  model manifest sha 3baa52f40876b78dab7e9428f2e488ca2ae3ed6b3d813df17f72e15a61fc516a

ANALYZER_PROVENANCE_FREEZE = PASS
{
 "morpheme_density_denominator": "eojeol_count",
 "particle_tag_rule": "base_tag.startswith(\"J\")",
 "ending_tag_rule": "base_tag.startswith(\"E\")",
 "deriv_affix_rule": "base_tag in {XSN, XSV, XSA} (XSA-I / XSA-R 등 irregular variant 포함)",
 "zero_morpheme_policy": "pair를 유지하고 analysis_warning_flag=true를 기록한다. 분모가 morpheme_count인 ratio는 null이며 대체 분모를 쓰지 않는다."
}


## 4 — SSOT §12.3 schema conformance and independent recomputation

The counts and ratios stored in the artifact are re-derived **from the stored `morpheme_sequence`**
rather than trusted. This is an *independent validation*: the sequence is the raw analyzer output, so
re-deriving the buckets from it in SQL exercises the mapping against the stored data without calling
the pipeline that wrote it.

In [6]:
D03_REQUIRED = ["morph_measurement_id", "pair_id", "analyzer_name", "analyzer_package_version",
                "analyzer_model_version", "analyzer_config_hash", "morpheme_sequence",
                "morpheme_count", "particle_count", "ending_count", "deriv_affix_count",
                "analysis_warning_flag"]
present = {row[0] for row in CON.execute(f"DESCRIBE SELECT * FROM {M}").fetchall()}
missing = [f for f in D03_REQUIRED if f not in present]
print(f"SSOT §12.3 required fields present: {len(D03_REQUIRED) - len(missing)}/{len(D03_REQUIRED)}"
      f"   missing {missing or '[]'}")
assert not missing

RECOMPUTE_SQL = (
    "WITH derived AS ("
    "  SELECT morpheme_count, particle_count, ending_count, deriv_affix_count, eojeol_count,"
    "         morpheme_density, particle_ratio, ending_ratio, deriv_affix_ratio,"
    "         function_morpheme_ratio, analysis_warning_flag,"
    "         length(morpheme_sequence) AS seq_len,"
    "         (SELECT count(*) FROM unnest(morpheme_sequence) AS t(x)"
    "            WHERE starts_with(split_part(x.pos, '-', 1), 'J')) AS re_particle,"
    "         (SELECT count(*) FROM unnest(morpheme_sequence) AS t(x)"
    "            WHERE starts_with(split_part(x.pos, '-', 1), 'E')) AS re_ending,"
    "         (SELECT count(*) FROM unnest(morpheme_sequence) AS t(x)"
    "            WHERE split_part(x.pos, '-', 1) IN ('XSN','XSV','XSA')) AS re_deriv"
    f"  FROM {M})"
    " SELECT"
    "   sum((seq_len <> morpheme_count)::INT) AS seq_len_mismatch,"
    "   sum((re_particle <> particle_count)::INT) AS particle_mismatch,"
    "   sum((re_ending <> ending_count)::INT) AS ending_mismatch,"
    "   sum((re_deriv <> deriv_affix_count)::INT) AS deriv_mismatch,"
    "   sum((abs(morpheme_density - morpheme_count::DOUBLE / eojeol_count) > 1e-12)::INT)"
    "     AS density_13_6_violation,"
    "   sum((morpheme_count > 0 AND"
    "        abs(particle_ratio - particle_count::DOUBLE / morpheme_count) > 1e-12)::INT)"
    "     AS particle_ratio_13_7_violation,"
    "   sum((morpheme_count > 0 AND"
    "        abs(ending_ratio - ending_count::DOUBLE / morpheme_count) > 1e-12)::INT)"
    "     AS ending_ratio_13_7_violation,"
    "   sum((morpheme_count > 0 AND"
    "        abs(function_morpheme_ratio -"
    "            (particle_count + ending_count)::DOUBLE / morpheme_count) > 1e-12)::INT)"
    "     AS function_ratio_violation,"
    "   sum(((morpheme_count = 0) <> analysis_warning_flag)::INT) AS warning_contract_violation,"
    "   sum((morpheme_count = 0 AND particle_ratio IS NOT NULL)::INT) AS null_ratio_violation,"
    "   sum((eojeol_count <= 0)::INT) AS eojeol_nonpositive"
    " FROM derived")
recompute = CON.execute(RECOMPUTE_SQL).fetchdf().iloc[0]
for key, value in recompute.items():
    print(f"  {key:34s} {int(value):,}")
assert all(int(v) == 0 for v in recompute.to_dict().values())  # Series.values는 속성이라 호출 불가
print("\nD03_INDEPENDENT_RECOMPUTATION = PASS (0 mismatches over the full cohort)")

cross = CON.execute(f"SELECT count(*) FROM {M} m JOIN {R} r USING (pair_id)"
                    f" WHERE m.eojeol_count <> r.ko_eojeol_count").fetchone()[0]
print(f"MORPH.eojeol_count != REP_v002.ko_eojeol_count : {cross}")
assert cross == 0

SSOT §12.3 required fields present: 12/12   missing []


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  seq_len_mismatch                   0
  particle_mismatch                  0
  ending_mismatch                    0
  deriv_mismatch                     0
  density_13_6_violation             0
  particle_ratio_13_7_violation      0
  ending_ratio_13_7_violation        0
  function_ratio_violation           0
  warning_contract_violation         0
  null_ratio_violation               0
  eojeol_nonpositive                 0

D03_INDEPENDENT_RECOMPUTATION = PASS (0 mismatches over the full cohort)


MORPH.eojeol_count != REP_v002.ko_eojeol_count : 0


## 5 — M2 effect: irregular derivational affixes

The pre-correction mapping matched `XSN`/`XSV`/`XSA` exactly and therefore dropped every irregular
variant such as `XSA-I`. Counting both mappings over the same stored sequences isolates exactly what
the fix changed, and shows it changed nothing else.

In [7]:
M2_SQL = (
    f"WITH tags AS (SELECT x.pos AS tag FROM {M}, unnest(morpheme_sequence) AS t(x))"
    " SELECT"
    "   (SELECT count(*) FROM tags WHERE split_part(tag,'-',1) IN ('XSN','XSV','XSA'))"
    "     AS base_mapping,"
    "   (SELECT count(*) FROM tags WHERE tag IN ('XSN','XSV','XSA')) AS exact_mapping,"
    "   (SELECT count(*) FROM tags WHERE tag LIKE 'XS%-%') AS irregular_affix")
m2 = CON.execute(M2_SQL).fetchdf().iloc[0]
delta = int(m2["base_mapping"]) - int(m2["exact_mapping"])
print(f"  base mapping  (current, irregulars counted) : {int(m2['base_mapping']):,}")
print(f"  exact mapping (pre-M2)                      : {int(m2['exact_mapping']):,}")
print(f"  delta                                       : {delta:,}")
print(f"  independently counted irregular affixes     : {int(m2['irregular_affix']):,}")
assert delta == int(m2["irregular_affix"]), "delta must equal the irregular-affix count exactly"
print("\nM2_EFFECT_CONSISTENT = PASS (the delta is exactly the irregular affixes and nothing else)")

irregular = CON.execute(
    f"SELECT x.pos AS tag, count(*) AS n FROM {M}, unnest(morpheme_sequence) AS t(x)"
    " WHERE x.pos LIKE '%-%' GROUP BY tag ORDER BY n DESC").fetchdf()
print("\nirregular tags over the full cohort:")
print(irregular.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  base mapping  (current, irregulars counted) : 6,712,268
  exact mapping (pre-M2)                      : 6,683,776
  delta                                       : 28,492
  independently counted irregular affixes     : 28,492

M2_EFFECT_CONSISTENT = PASS (the delta is exactly the irregular affixes and nothing else)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


irregular tags over the full cohort:
  tag      n
 VA-I 286471
 VV-R 176697
 VV-I  83316
XSA-I  28492
 VA-R   3221
 VX-R     46


## 6 — Distribution and analyzer failure rate

SSOT §31 G4 requires the analyzer failure rate to be reported.

In [8]:
DIST_SQL = (
    "SELECT count(*) AS n,"
    "  sum(analysis_warning_flag::INT) AS warning_rows,"
    "  sum((morpheme_count = 0)::INT) AS zero_morpheme_rows,"
    "  sum(morpheme_count) AS total_morphemes, sum(particle_count) AS total_particles,"
    "  sum(ending_count) AS total_endings, sum(deriv_affix_count) AS total_deriv,"
    "  min(eojeol_count) AS eojeol_min, median(eojeol_count) AS eojeol_median,"
    "  max(eojeol_count) AS eojeol_max,"
    "  min(morpheme_density) AS density_min, median(morpheme_density) AS density_median,"
    "  quantile_cont(morpheme_density, 0.99) AS density_p99, max(morpheme_density) AS density_max,"
    "  median(particle_ratio) AS particle_median, median(ending_ratio) AS ending_median,"
    "  median(deriv_affix_ratio) AS deriv_median,"
    "  median(function_morpheme_ratio) AS function_median"
    f" FROM {M}")
dist = CON.execute(DIST_SQL).fetchdf().iloc[0]
print(f"  rows                {int(dist['n']):,}")
print(f"  analyzer failures   {int(dist['warning_rows']):,}  "
      f"(rate {int(dist['warning_rows']) / int(dist['n']):.6f})")
print(f"  zero-morpheme rows  {int(dist['zero_morpheme_rows']):,}")
print(f"  morphemes {int(dist['total_morphemes']):,} · particles {int(dist['total_particles']):,}"
      f" · endings {int(dist['total_endings']):,} · deriv {int(dist['total_deriv']):,}")
print(f"  eojeol   min {int(dist['eojeol_min'])} median {dist['eojeol_median']:.1f} "
      f"max {int(dist['eojeol_max'])}")
print(f"  density  min {dist['density_min']:.4f} median {dist['density_median']:.4f} "
      f"p99 {dist['density_p99']:.4f} max {dist['density_max']:.4f}")
print(f"  medians  particle {dist['particle_median']:.4f} ending {dist['ending_median']:.4f} "
      f"deriv {dist['deriv_median']:.4f} function {dist['function_median']:.4f}")

  rows                3,835,988
  analyzer failures   0  (rate 0.000000)
  zero-morpheme rows  0
  morphemes 92,652,231 · particles 15,156,896 · endings 15,824,538 · deriv 6,712,268
  eojeol   min 1 median 9.0 max 158
  density  min 0.5000 median 2.2143 p99 4.0000 max 30.0000
  medians  particle 0.1579 ending 0.1754 deriv 0.0652 function 0.3421


### Caveat carried forward to NB07/NB08 (Claude-B, R3)

The density maximum is an analyzer artifact of very short input, not a linguistic signal. Rows with
`eojeol_count = 1` are whitespace-free strings, so the §13.6 denominator is 1 and the density equals
the raw morpheme count. Claude-B classified these as `PLAUSIBLE_EXTREME`, not an implementation
defect. Distribution and extreme-case panels downstream must treat them that way.

In [9]:
SHORT_SQL = (
    "SELECT sum((eojeol_count = 1)::INT) AS eojeol_one, count(*) AS n,"
    "  max(morpheme_density) FILTER (WHERE eojeol_count = 1) AS density_max_at_one,"
    "  max(morpheme_density) FILTER (WHERE eojeol_count > 1) AS density_max_above_one,"
    "  max(particle_ratio) FILTER (WHERE eojeol_count = 1) AS particle_max_at_one"
    f" FROM {M}")
short = CON.execute(SHORT_SQL).fetchdf().iloc[0]
print(f"  eojeol_count = 1 rows      {int(short['eojeol_one']):,} "
      f"({int(short['eojeol_one']) / int(short['n']):.4%})")
print(f"  max density where eojeol=1 {short['density_max_at_one']:.4f}")
print(f"  max density where eojeol>1 {short['density_max_above_one']:.4f}")
print(f"  max particle_ratio at 1    {short['particle_max_at_one']:.4f}")
print("\nPLAUSIBLE_EXTREME — a reporting caveat for NB07/NB08, not a defect.")

  eojeol_count = 1 rows      42,096 (1.0974%)
  max density where eojeol=1 30.0000
  max density where eojeol>1 9.6667
  max particle_ratio at 1    0.6000

PLAUSIBLE_EXTREME — a reporting caveat for NB07/NB08, not a defect.


## 7 — Human audit lineage

The Director's manual audit of the corrected pipeline. The superseded 85-row sanity sample is not
used as evidence anywhere in this notebook.

In [10]:
audit = json.loads((PROJECT_ROOT / "outputs/manifests/MORPHOLOGY_AUDIT_100_MANIFEST_v001.json"
                    ).read_text(encoding="utf-8"))
print(f"  sampling key      {audit['sampling_key']['path']}")
print(f"  n / target        {audit['selected_n']} / {audit['target_n']}")
print(f"  exact set equality {audit['exact_set_equality_100']}")
print(f"  coverage domain   {audit['coverage']['domain']}")
print(f"  coverage length   {audit['coverage']['length_stratum']}")
print(f"  irregular-affix   {audit['irregular_affix_cases']}")
print("\nDirector verdict O=94 / X=6 with implementation defect 0 (Claude-B §N).")
print("The 6% is NOT a population error rate: the sample deliberately over-represents")
print("domain balance, metric extremes, structural stress and irregular-affix cases.")

  sampling key      outputs/manual_audit/MORPHOLOGY_AUDIT_100_SAMPLING_KEY_v001.csv
  n / target        100 / 100
  exact set equality True
  coverage domain   {'dialogue': 17, 'general': 35, 'other': 35, 'technology': 13}
  coverage length   {'Q1': 53, 'Q2': 10, 'Q3': 9, 'Q4': 9, 'Q5': 19}
  irregular-affix   4

Director verdict O=94 / X=6 with implementation defect 0 (Claude-B §N).
The 6% is NOT a population error rate: the sample deliberately over-represents
domain balance, metric extremes, structural stress and irregular-affix cases.


## 8 — Canonical summary

In [11]:
summary = {
    "notebook": "notebooks/04_morphology_features.ipynb",
    "phase": "Phase 3 — D-03 Morphology Measurement",
    "canonical_artifact": {k: MORPH[k] for k in
                           ("name", "path", "sha256", "row_count", "column_count", "pair_set_md5")},
    "superseded": {"MORPH_FEATURES_PILOT_v001": pilot["status"],
                   "sanity_sample_85_row": "SUPERSEDED_BY_CONFORMANCE_DEFECT"},
    "analyzer_provenance_freeze": "PASS",
    "d03_schema_conformance": "PASS",
    "d03_independent_recomputation": "PASS",
    "m2_effect_consistent": "PASS",
    "analyzer_failure_rows": int(dist["warning_rows"]),
    "irregular_affix_delta": delta,
    "gate": ("G4_MORPHOLOGY_INTEGRITY_PASS adjudicated by Claude-B at "
             f"{ADJUDICATION_COMMIT[:12]}; this notebook reproduces the evidence, "
             "it does not re-adjudicate the gate"),
    "rebuild_performed": False,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
CON.close()

{
  "notebook": "notebooks/04_morphology_features.ipynb",
  "phase": "Phase 3 — D-03 Morphology Measurement",
  "canonical_artifact": {
    "name": "MORPH_FEATURES_KIWI_v001",
    "path": "data/registry/MORPH_FEATURES_KIWI_v001.parquet",
    "sha256": "0fe5bd74e3993a7141c5c33ea78e71b2c66e3ecd296544bde2615acb43e50f7d",
    "row_count": 3835988,
    "column_count": 19,
    "pair_set_md5": "d9660d654ee449e4d0c23a0070225274"
  },
  "superseded": {
    "MORPH_FEATURES_PILOT_v001": "SUPERSEDED_BY_CONFORMANCE_DEFECT",
    "sanity_sample_85_row": "SUPERSEDED_BY_CONFORMANCE_DEFECT"
  },
  "analyzer_provenance_freeze": "PASS",
  "d03_schema_conformance": "PASS",
  "d03_independent_recomputation": "PASS",
  "m2_effect_consistent": "PASS",
  "analyzer_failure_rows": 0,
  "irregular_affix_delta": 28492,
  "gate": "G4_MORPHOLOGY_INTEGRITY_PASS adjudicated by Claude-B at 441d5802bfeb; this notebook reproduces the evidence, it does not re-adjudicate the gate",
  "rebuild_performed": false
}


---

`MORPH_FEATURES_KIWI_v001` is the canonical D-03 artifact. The N=1,000 pilot and its 85-row sanity
sample remain historical and may not be cited as G4 evidence.